In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

import osmnx as ox

import libpysal
import momepy as mm

from tqdm.auto import tqdm

import scipy as sp
from scipy.cluster import hierarchy
from sklearn import preprocessing
from sklearn.mixture import GaussianMixture

import mapclassify
import matplotlib.pyplot as plt
import seaborn as sns

# **COMPONENTS**

## **Brooklyn Buildings & Streets**

In [ ]:
boroughs = gpd.read_file('data/boroughs.zip')
boroughs = gpd.GeoDataFrame(boroughs[['boro_name', 'geometry']]).to_crs('EPSG:26918')

In [ ]:
buildings = gpd.read_file('data/buildings.zip')

buildings = buildings[['heightroof', 'geometry']]
buildings = buildings.rename(columns={'heightroof':'height'})
buildings = buildings.dropna()

buildings = gpd.GeoDataFrame(buildings)
buildings = buildings.to_crs('EPSG:26918')
buildings = buildings.sjoin(boroughs, how='left', predicate='within')

buildings_bk = gpd.GeoDataFrame(buildings[buildings.boro_name == 'Brooklyn'][['height', 'geometry']])
buildings_bk = buildings_bk.set_crs('EPSG:26918')

buildings_bk['uID'] = mm.unique_id(buildings_bk)

In [ ]:
streets = ox.graph_from_place('New York, New York', network_type='drive')
streets = ox.utils_graph.graph_to_gdfs(streets, nodes=False, edges=True, node_geometry=False, fill_edge_geometry=True)
streets = gpd.GeoDataFrame(streets[['geometry']])
streets = streets.to_crs('EPSG:26918')

streets = streets.sjoin(boroughs, how='left', predicate='within')

streets_bk = gpd.GeoDataFrame(streets[streets.boro_name == 'Brooklyn'][['geometry']]).reset_index(drop=True)
streets_bk = streets_bk.set_crs('EPSG:26918')

streets_bk = mm.remove_false_nodes(streets_bk)
streets_bk.geometry = mm.close_gaps(streets_bk, .25)
streets_bk = mm.extend_lines(streets_bk, tolerance=.2)

In [ ]:
streets_bk['nID'] = mm.unique_id(streets_bk)
buildings_bk['nID'] = mm.get_network_id(buildings_bk, streets_bk, 'nID', min_size=300, verbose=False)
buildings_bk = buildings_bk.dropna()

buildings_bk.to_file('data/buildings_bk.shp')
streets_bk.to_file('data/streets_bk.shp')

## **Brooklyn Tessellation**

In [ ]:
limit_bk = mm.buffered_limit(buildings_bk)
enclosures_bk = mm.enclosures(streets_bk, limit=gpd.GeoSeries([limit_bk]))

tessellation_bk = mm.Tessellation(buildings_bk, unique_id='uID', enclosures=enclosures_bk, use_dask=True).tessellation
tessellation_bk = tessellation_bk.dropna()

In [ ]:
tessellation_bk = tessellation_bk.merge(buildings_bk[['uID', 'nID']], on='uID', how='left')
tessellation_bk.to_file('data/tessellation_bk.shp')

## **Brooklyn Blocks**

In [ ]:
limit_bk = mm.buffered_limit(buildings_bk)
extended_bk = mm.extend_lines(streets_bk, tolerance=120, target=gpd.GeoSeries([limit_bk.boundary]), barrier=buildings_bk)

In [ ]:
blocks_bk = mm.Blocks(tessellation_bk, edges=extended_bk, buildings=buildings_bk, id_name='bID', unique_id='uID')

blocks_bk_df = blocks_bk.blocks
blocks_bk_df.to_file('data/blocks_bk.shp')

In [ ]:
buildings_bk['bID'] = blocks_bk.buildings_id.values
tessellation_bk['bID'] = blocks_bk.tessellation_id.values

buildings_bk.to_file('data/buildings_bk.shp')
tessellation_bk.to_file('data/tessellation_bk.shp')

# **MORPHOMETRICS**

In [ ]:
buildings_bk['buildingHeight'] = buildings_bk['height'] * 0.304800
buildings_bk['buildingArea'] = mm.Area(buildings_bk).series
buildings_bk['buildingFloorArea'] = (buildings_bk['buildingHeight'] / 3.5) * buildings_bk['buildingArea']
buildings_bk['buildingVolume'] = mm.Volume(buildings_bk, 'buildingHeight', 'buildingArea').series
buildings_bk['buildingPerimeter'] = mm.Perimeter(buildings_bk).series
buildings_bk['buildingCourtyardArea'] = mm.CourtyardArea(buildings_bk, 'buildingArea').series
buildings_bk['buildingCircularCompactness'] = mm.CircularCompactness(buildings_bk, 'buildingArea').series
buildings_bk['buildingElongation'] = mm.Elongation(buildings_bk).series
buildings_bk['buildingSquareness'] = mm.Squareness(buildings_bk, verbose=False).series
buildings_bk['buildingFormFactor'] = mm.FormFactor(buildings, 'buildingVolume', 'buildingArea', 'buildingHeight').series
buildings_bk['buildingVolumeFacadeRatio'] = mm.VolumeFacadeRatio(buildings_bk, 'buildingHeight', 'buildingArea', 'buildingPerimeter').series
buildings_bk['buildingCorners'] = mm.Corners(buildings_bk, verbose=False).series
buildings_bk['buildingEquivalentRectangularIndex'] = mm.EquivalentRectangularIndex(buildings_bk, 'buildingArea', 'buildingPerimeter').series
buildings_bk['buildingSharedWallsRatio'] = mm.SharedWallsRatio(buildings_bk, 'buildingPerimeter').series

centroid_corners = mm.CentroidCorners(buildings_bk, verbose=False)
buildings_bk['buildingCentroidCornersMean'] = centroid_corners.mean
buildings_bk['buildingCentroidCornersStD'] = centroid_corners.std

tessellation_bk['tessArea'] = mm.Area(tessellation_bk).series
tessellation_bk['tessCompactnessWeightedAxis'] = mm.CompactnessWeightedAxis(tessellation_bk)
tessellation_bk['tessCircularCompactness'] = mm.CircularCompactness(tessellation_bk, 'tessArea').series
tessellation_bk['tessLongestAxisLength'] = mm.LongestAxisLength(tessellation_bk).series
tessellation_bk['tessEquivalentRectangularIndex'] = mm.EquivalentRectangularIndex(tessellation_bk, 'tessArea').series

blocks_bk = blocks_bk_df
blocks_bk['blockArea'] = mm.Area(blocks_bk).series
blocks_bk['blockPerimeter'] = mm.Perimeter(blocks_bk).series
blocks_bk['blockCircularCompactness'] = mm.CircularCompactness(blocks_bk, 'blockArea').series
blocks_bk['blockEquivalentRectangularIndex'] = mm.EquivalentRectangularIndex(blocks_bk, 'blockArea', 'blockPerimeter').series
blocks_bk['blockCompactnessWeightedAxis'] = mm.CompactnessWeightedAxis(blocks_bk, 'blockArea', 'blockPerimeter').series
blocks_bk['blockOrientation'] = mm.Orientation(blocks_bk, verbose=False).series

In [ ]:
buildings_bk['buildingOrientation'] = mm.Orientation(buildings_bk, verbose=False).series
tessellation_bk['tessOrientation'] = mm.Orientation(tessellation_bk, verbose=False).series
buildings_bk['buildingCellAlignment'] = mm.CellAlignment(buildings_bk, tessellation_bk, 'buildingOrientation', 'tessOrientation', 'uID', 'uID').series

tessellation_bk['tessStreetAlignment'] = mm.StreetAlignment(tessellation_bk, streets_bk, 'tessOrientation', 'nID').series
buildings_bk['buildingStreetAlignment'] = mm.StreetAlignment(buildings_bk, streets_bk, 'buildingOrientation', 'nID').series

In [ ]:
profile = mm.StreetProfile(streets_bk, buildings_bk, heights='buildingHeight', distance=3)
streets_bk['streetWidth'] = profile.w
streets_bk['streetHeight'] = profile.h
streets_bk['streetProfile'] = profile.p
streets_bk['streetOpenness'] = profile.o
streets_bk['streetWidthDeviations'] = profile.wd
streets_bk['streetHeightDeviations'] = profile.hd

In [ ]:
tessellation_bk['tessCoveredAreaRatio'] = mm.AreaRatio(tessellation_bk, buildings_bk, 'tessArea', 'buildingArea', 'uID').series

In [ ]:
streets_bk['streetPerimeter'] = mm.Perimeter(streets_bk).series
streets_bk['streetLinearity'] = mm.Linearity(streets_bk).series
streets_bk["streetArea"] = mm.Reached(streets_bk, tessellation_bk, 'nID', 'nID', mode='sum', values='tessArea').series
streets_bk["streetBuildingCount"] = mm.Count(streets_bk, buildings_bk, 'nID', 'nID', weighted=True).series

In [ ]:
queen_tessellation = libpysal.weights.contiguity.Queen.from_dataframe(tessellation_bk, idVariable='uID', silence_warnings=True)
queen_buildings = libpysal.weights.contiguity.Queen.from_dataframe(buildings_bk, silence_warnings=True)
queen_blocks = libpysal.weights.contiguity.Queen.from_dataframe(blocks_bk, ids='bID', silence_warnings=True)
queen_streets = libypysal.weights.contiguity.Queen.from_dataframe(streets_bk, silence_warnings=True)

queen_k3 = mm.sw_high(k=3, weights=queen_tessellation)

In [ ]:
buildings_bk['buildingAlignment'] = mm.Alignment(buildings_bk, queen_tessellation, 'uID', 'buildingOrientation', verbose=False).series
buildings_bk['buildingNeighborDistance'] = mm.NeighborDistance(buildings_bk, queen_tessellation, 'uID', verbose=False).series
buildings_bk['buildingCourtyards'] = mm.Courtyards(buildings_bk, spatial_weights=queen_buildings, verbose=False).series
buildings_bk['buildingPerimeterWall'] = mm.PerimeterWall(buildings_bk, queen_buildings, verbose=False).series

tessellation_bk['tessNeighbors'] = mm.Neighbors(tessellation_bk, queen_tessellation, 'uID', weighted=True, verbose=False).series
tessellation_bk['tessCoveredArea'] = mm.CoveredArea(tessellation_bk, queen_tessellation, 'uID', verbose=False).series

blocks_bk['blockNeighbors'] = mm.Neighbors(blocks_bk, queen_blocks, 'bID', weighted=True, verbose=False).series
blocks_bk['blockCount'] = mm.Count(blocks_bk, buildings_bk, 'bID', 'bID', weighted=True).series

In [ ]:
buildings_bk['buildingMeanInterbuildingDistance'] = mm.MeanInterbuildingDistance(buildings_bk, queen_tessellation, 'uID', queen_k3, verbose=False).series
buildings_bk['buildingBuildingAdjacency'] = mm.BuildingAdjacency(buildings_bk, queen_k3, 'uID', queen_buildings, verbose=False).series

In [ ]:
buildings_bk['height'] = buildings_bk['height'] * 0.304800
buildings_bk['floor_area'] = (buildings_bk["height"] / 3.5) * buildings.area

In [ ]:
buildings_bk['sdbAre'] = mm.Area(buildings_bk).series
buildings_bk['sdbVol'] = mm.Volume(buildings_bk, 'height', 'sdbAre').series
buildings_bk['sdbPer'] = mm.Perimeter(buildings_bk).series
buildings_bk['sdbCoA'] = mm.CourtyardArea(buildings_bk, 'sdbAre').series

buildings_bk['ssbFoF'] = mm.FormFactor(buildings_bk, 'sdbVol', 'sdbAre', 'height').series
buildings_bk['ssbVFR'] = mm.VolumeFacadeRatio(buildings_bk, 'height', 'sdbVol', 'sdbPer').series
buildings_bk['ssbCCo'] = mm.CircularCompactness(buildings_bk, 'sdbAre').series
buildings_bk['ssbCor'] = mm.Corners(buildings_bk, verbose=False).series
buildings_bk['ssbSqu'] = mm.Squareness(buildings_bk, verbose=False).series
buildings_bk['ssbERI'] = mm.EquivalentRectangularIndex(buildings_bk, 'sdbAre', 'sdbPer').series
buildings_bk['ssbElo'] = mm.Elongation(buildings_bk).series

In [ ]:
cencon = mm.CentroidCorners(buildings_bk, verbose=False)
buildings_bk['ssbCCM'] = cencon.mean
buildings_bk['ssbCCD'] = cencon.std

In [ ]:
tessellation_bk['sdcLAL'] = mm.LongestAxisLength(tessellation_bk).series
tessellation_bk['sdcAre'] = mm.Area(tessellation_bk).series
tessellation_bk['sscCCo'] = mm.CircularCompactness(tessellation_bk, 'sdcAre').series
tessellation_bk['sscERI'] = mm.EquivalentRectangularIndex(tessellation_bk, 'sdcAre').series

In [ ]:
buildings_bk

In [ ]:
tessellation_bk.iloc[255555, :].geometry

In [ ]:
buildings_bk['stbOri'] = mm.Orientation(buildings_bk, verbose=False).series
tessellation_bk['stcOri'] = mm.Orientation(tessellation_bk, verbose=False).series
buildings_bk['stbCeA'] = mm.CellAlignment(buildings_bk, tessellation_bk, 'stbOri', 'stcOri', 'uID', 'uID').series

In [ ]:
buildings_bk["mtbSWR"] = mm.SharedWallsRatio(buildings_bk, "sdbPer").series
 
queen_tessellation_bk = libpysal.weights.contiguity.Queen.from_dataframe(tessellation_bk, idVariable="uID", silence_warnings=True)
 
buildings_bk["mtbAli"] = mm.Alignment(buildings_bk, queen_tessellation_bk, "uID", "stbOri", verbose=False).series
buildings_bk["mtbNDi"] = mm.NeighborDistance(buildings_bk, queen_tessellation_bk, "uID", verbose=False).series
tessellation_bk["mtcWNe"] = mm.Neighbors(tessellation_bk, queen_tessellation_bk, "uID", weighted=True, verbose=False).series
tessellation_bk["mdcAre"] = mm.CoveredArea(tessellation_bk, queen_tessellation_bk, "uID", verbose=False).series

In [ ]:
queen_buildings_bk = libpysal.weights.contiguity.Queen.from_dataframe(buildings_bk, silence_warnings=True)
 
buildings_bk["libNCo"] = mm.Courtyards(buildings_bk, spatial_weights=queen_buildings_bk, verbose=False).series
buildings_bk["ldbPWL"] = mm.PerimeterWall(buildings_bk, queen_buildings_bk, verbose=False).series

In [ ]:
blocks_bk = blocks_bk_df

In [ ]:
blocks_bk["ldkAre"] = mm.Area(blocks_bk).series
blocks_bk["ldkPer"] = mm.Perimeter(blocks_bk).series
blocks_bk["lskCCo"] = mm.CircularCompactness(blocks_bk, "ldkAre").series
blocks_bk["lskERI"] = mm.EquivalentRectangularIndex(blocks_bk, "ldkAre", "ldkPer").series
blocks_bk["lskCWA"] = mm.CompactnessWeightedAxis(blocks_bk, "ldkAre", "ldkPer").series
blocks_bk["ltkOri"] = mm.Orientation(blocks_bk, verbose=False).series

In [ ]:
queen_blocks_bk = libpysal.weights.contiguity.Queen.from_dataframe(blocks_bk, ids="bID", silence_warnings=True)
 
blocks_bk["ltkWNB"] = mm.Neighbors(blocks_bk, queen_blocks_bk, "bID", weighted=True, verbose=False).series
blocks_bk["likWBB"] = mm.Count(blocks_bk, buildings_bk, "bID", "bID", weighted=True).series

In [ ]:
queen = mm.sw_high(k=3, weights=queen_tessellation_bk)

buildings_bk['ltbIBD'] = mm.MeanInterbuildingDistance(buildings_bk, queen_tessellation_bk, 'uID', queen, verbose=False).series
buildings_bk['ltcBuA'] = mm.BuildingAdjacency(buildings_bk, queen, 'uID', queen_buildings_bk, verbose=False).series

In [ ]:
tessellation_bk = tessellation_bk.merge(buildings_bk[['floor_area', 'uID']], on='uID', how='left')
tessellation_bk['licGDe'] = mm.Density(tessellation_bk, 'floor_area', queen, 'uID', 'sdcAre').series
tessellation_bk = tessellation_bk.drop(columns='floor_area')
tessellation_bk['ltcWRB'] = mm.BlocksCount(tessellation_bk, 'bID', queen, 'uID', verbose=False).series
tessellation_bk['sicCAR'] = mm.AreaRatio(tessellation_bk, buildings_bk, 'sdcAre', 'sdbAre', 'uID').series
tessellation_bk['sicFAR'] = mm.AreaRatio(tessellation_bk, buildings_bk, 'sdcAre', 'floor_area', 'uID').series

In [ ]:
streets_bk["sdsLen"] = mm.Perimeter(streets_bk).series
tessellation_bk["stcSAl"] = mm.StreetAlignment(tessellation_bk, streets_bk, "stcOri", "nID").series
buildings_bk["stbSAl"] = mm.StreetAlignment(buildings_bk, streets_bk, "stbOri", "nID").series

In [ ]:
profile = mm.StreetProfile(streets_bk, buildings_bk, heights='height', distance=3)
streets_bk["sdsSPW"] = profile.w
streets_bk["sdsSPH"] = profile.h
streets_bk["sdsSPR"] = profile.p
streets_bk["sdsSPO"] = profile.o
streets_bk["sdsSWD"] = profile.wd
streets_bk["sdsSHD"] = profile.hd

In [ ]:
streets_bk["sssLin"] = mm.Linearity(streets_bk).series
streets_bk["sdsAre"] = mm.Reached(streets_bk, tessellation_bk, "nID", "nID", mode="sum", values="sdcAre").series
streets_bk["sisBpM"] = mm.Count(streets_bk, buildings_bk, "nID", "nID", weighted=True).series

In [ ]:
queen_streets_bk = libpysal.weights.contiguity.Queen.from_dataframe(streets_bk, silence_warnings=True)
 
streets_bk["misRea"] = mm.Reached(streets_bk, tessellation_bk, "nID", "nID", spatial_weights=queen_streets_bk, mode="count", verbose=False).series
streets_bk["mdsAre"] = mm.Reached(streets_bk, tessellation_bk, "nID", "nID", spatial_weights=queen_streets_bk, mode="sum", verbose=False).series

In [ ]:
graph = mm.gdf_to_nx(streets_bk)
graph = mm.node_degree(graph)
graph = mm.subgraph(
    graph,
    radius=5,
    meshedness=True,
    cds_length=False,
    mode="sum",
    degree="degree",
    length="mm_len",
    mean_node_degree=False,
    proportion={0: True, 3: True, 4: True},
    cyclomatic=False,
    edge_node_ratio=False,
    gamma=False,
    local_closeness=True,
    closeness_weight="mm_len", 
    verbose=False
)
graph = mm.cds_length(graph, radius=3, name="ldsCDL", verbose=False)
graph = mm.clustering(graph, name="xcnSCl")
graph = mm.mean_node_dist(graph, name="mtdMDi", verbose=False)
 
nodes, edges, sw = mm.nx_to_gdf(graph, spatial_weights=True)

In [ ]:
edges_w3 = mm.sw_high(k=3, gdf=edges)
edges["ldsMSL"] = mm.SegmentsLength(edges, spatial_weights=edges_w3, mean=True, verbose=False).series
 
edges["ldsRea"] = mm.Reached(edges, tessellation_bk, "nID", "nID", spatial_weights=edges_w3, verbose=False).series
edges["ldsRea"] = mm.Reached(edges, tessellation_bk, "nID", "nID", spatial_weights=edges_w3, mode="sum", values="sdcAre", verbose=False).series
 
nodes_w5 = mm.sw_high(k=5, weights=sw)
nodes["lddNDe"] = mm.NodeDensity(nodes, edges, nodes_w5, verbose=False).series
nodes["linWID"] = mm.NodeDensity(nodes, edges, nodes_w5, weighted=True, node_degree="degree", verbose=False).series
 
buildings_bk["nodeID"] = mm.get_node_id(buildings_bk, nodes, edges, "nodeID", "nID")
tessellation_bk = tessellation_bk.merge(buildings_bk[["uID", "nodeID"]], on="uID", how="left")
 
nodes_w3 = mm.sw_high(k=3, weights=sw)
 
nodes["lddRea"] = mm.Reached(nodes, tessellation_bk, "nodeID", "nodeID", nodes_w3, verbose=False).series
nodes["lddARe"] = mm.Reached(nodes, tessellation_bk, "nodeID", "nodeID", nodes_w3, mode="sum", values="sdcAre", verbose=False).series
 
nodes["sddAre"] = mm.Reached(nodes, tessellation_bk, "nodeID", "nodeID", mode="sum", values="sdcAre", verbose=False).series
nodes["midRea"] = mm.Reached(nodes, tessellation_bk, "nodeID", "nodeID", spatial_weights=sw, verbose=False).series
nodes["midAre"] = mm.Reached(nodes, tessellation_bk, "nodeID", "nodeID", spatial_weights=sw, mode="sum", values="sdcAre", verbose=False).series
 
nodes.rename(
    columns={
        "degree": "mtdDeg",
        "meshedness": "lcdMes",
        "local_closeness": "lcnClo",
        "proportion_3": "linP3W",
        "proportion_4": "linP4W",
        "proportion_0": "linPDE",
    }, inplace=True
)

In [ ]:
merged = tessellation_bk.merge(buildings_bk.drop(columns=['nID', 'bID', 'nodeID', 'geometry']), on='uID')
merged = merged.merge(blocks_bk.drop(columns='geometry'), on='bID', how='left')
merged = merged.merge(edges.drop(columns='geometry'), on='nID', how='left')
merged = merged.merge(nodes.drop(columns='geometry'), on='nodeID', how='left')

primary = merged.drop(columns=['nID', 'bID', 'nodeID', 'mm_len', 'cdsbool', 
                               'node_start', 'node_end', 'geometry', 'floor_area'])

In [ ]:
primary.to_csv('data/primary.csv')

# **SPATIALLY LAGGED MORPHOMETRICS**

In [ ]:
gdf = primary.set_index('uID')
spatial_weights = queen
unique_id = 'uID'

In [ ]:
means = {}
ranges = {}
theils = {}
simpsons = {}

for ch in gdf.columns:
    means[ch] = []
    ranges[ch] = []
    theils[ch] = []
    simpsons[ch] = []

In [ ]:
gdf = gdf.replace(np.inf, np.nan).fillna(0)  # normally does not happen, but to be sure
chars = gdf.columns

In [ ]:
gdf['lcdMes'] = gdf.apply(
            lambda row: row.lcdMes if row.lcdMes >= 0 else 0,
            axis=1,
        )  # normally does not happen, but to be sure

In [ ]:
def _theil(y):
    y = np.array(y)
    n = len(y)
    plus = y + np.finfo('float').tiny * (y == 0)  # can't have 0 values
    yt = plus.sum(axis=0)
    s = plus / (yt * 1.0)
    lns = np.log(n * s)
    slns = s * lns
    t = sum(slns)
    return t

In [ ]:
def _simpson_di(data):

    def p(n, N):
        if n == 0:
            return 0
        return float(n) / N

    N = sum(data.values())

    return sum(p(n, N) ** 2 for n in data.values() if n != 0)

In [ ]:
skewness = pd.DataFrame(index=chars)
for c in chars:
    skewness.loc[c, 'skewness'] = sp.stats.skew(gdf[c])
headtail = list(skewness.loc[skewness.skewness >= 1].index)
to_invert = skewness.loc[skewness.skewness <= -1].index

for inv in to_invert:
    gdf[inv + '_r'] = gdf[inv].max() - gdf[inv]
inverted = [x for x in gdf.columns if '_r' in x]
headtail = headtail + inverted
natural = [x for x in chars if x not in headtail]

In [ ]:
bins = {}
for c in headtail:
    bins[c] = mapclassify.HeadTailBreaks(gdf[c]).bins
for c in natural:
    bins[c] = mapclassify.gadf(gdf[c], method='NaturalBreaks')[1].bins

In [ ]:
for index in list(spatial_weights.neighbors.keys()):
    neighbours = [index]
    neighbours += spatial_weights.neighbors[index]
    
    subset = gdf.loc[neighbours]
    for ch in chars:
        values_list = subset[ch] 
        idec = mm.limit_range(values_list, rng=(10, 90))
        iquar = mm.limit_range(values_list, rng=(25, 75))
        
        means[ch].append(np.mean(iquar))
        ranges[ch].append(max(iquar) - min(iquar))
        theils[ch].append(_theil(idec))
        
        sample_bins = mapclassify.UserDefined(values_list, list(bins[ch]))
        counts = dict(zip(bins[ch], sample_bins.counts))
        simpsons[ch].append(_simpson_di(counts))

In [ ]:
contextual = {}
for ch in chars:
    contextual[ch + '_meanIQ3'] = means[ch]
    contextual[ch + '_rangeIQ3'] = ranges[ch]
    contextual[ch + '_theilID3'] = theils[ch]
    contextual[ch + '_simpson'] = simpsons[ch]

contextual = pd.DataFrame(contextual, index=gdf.index)

In [ ]:
contextual.to_csv('data/contextual.csv', index=False)

# **CLUSTERING**

In [ ]:
data = contextual.copy()

In [ ]:
x = data.values
scaler = preprocessing.StandardScaler()
cols = list(data.columns)
data[cols] = scaler.fit_transform(data[cols])

In [ ]:
bic = pd.DataFrame(columns=['n', 'bic', 'run'])
ix = 0

n_components_range = range(2, 8) # specify range you want to assess. 2-8 is sensible for the sample, not the whole city
gmmruns = 1  # specify how many times should each option be tried (more better, but takes a long time)

In [ ]:
data = data.fillna(0)
for n_components in n_components_range:
    for i in range(gmmruns):
        gmm = GaussianMixture(n_components=n_components, covariance_type="full", max_iter=200, n_init=1, verbose=1)
        fitted = gmm.fit(data)
        bicnum = gmm.bic(data)
        bic.loc[ix] = [n_components, bicnum, i]
        ix += 1

        print(n_components, i, "BIC:", bicnum)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 16))
sns.lineplot(ax=ax, x='n', y='bic', data=bic)
# plt.savefig('../files/complete_BIC.pdf')

In [ ]:
n = 5  # illustrative - always base the number on a reasonable estimation of the optimal number of components
n_init = 5  # more initialization, more stable clustering gets

gmm = GaussianMixture(n_components=n, covariance_type="full", max_iter=200, n_init=n_init, verbose=1)
fitted = gmm.fit(data)

In [ ]:
labels = gmm.predict(data)

In [ ]:
pd.Series(labels, index=data.index).to_csv('data/cluster_labels.csv')

In [ ]:
buildings_bk.plot(labels, categorical=True, cmap='tab20', legend=True, figsize=(20, 20))